# Airway Signal Lab: CNN/GRU/temporal-attention ablation

This notebook runs E0--E3 on the **original source-split prepared arrays only**. Synthetic obstruction classes remain signal-processing surrogates, not clinical labels. It does not alter the Streamlit TFLite model.

In [ ]:
# Colab environment: GPU is optional; the runner falls back to CPU.
!pip -q install 'tensorflow>=2.16,<2.19' scikit-learn matplotlib numpy
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Mount Drive, or use the upload cell below.
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/airway-signal-lab'  # change if needed
DATA_NPZ = '/content/drive/MyDrive/airway-training/prepared_source_split.npz'
MANIFEST_CSV = '/content/drive/MyDrive/airway-training/window_manifest.csv'
OUTPUT_DIR = f'{PROJECT}/experiments/hybrid_ablation'


In [ ]:
# Alternative to Drive: upload the required prepared archive and manifest.
# from google.colab import files
# files.upload()
# DATA_NPZ = '/content/prepared_source_split.npz'
# MANIFEST_CSV = '/content/window_manifest.csv'


## Required artifact contract

`prepared_source_split.npz` must contain `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test`. `X_*` must be normalized 32-band log-Mel tensors `(N, 32, frames)` or `(N, 32, frames, 1)` using **training data only**. Labels are fixed integer indices or one-hot columns in this order: normal, ambient, movable_synthetic, fixed_synthetic.

The CSV manifest must include `split`, `group_id`, and `class_name`, with each source group in exactly one split. Do not upload randomized existing windows without group provenance.

In [ ]:
# Copy only the source code needed for the experiment; no deployment model is changed.
%cd {PROJECT}
from experiments.hybrid_ablation import load_prepared_npz, validate_provenance, smoke_models
data = load_prepared_npz(DATA_NPZ)
provenance = validate_provenance(data, MANIFEST_CSV)
print({k: v.shape for k, v in data.items() if k.startswith(('X_', 'y_'))})
print(provenance)
print(smoke_models(tuple(data['X_train'].shape[1:])))


In [ ]:
# Fixed single-seed ablation. Checkpoints are selected on validation loss only.
from experiments.hybrid_ablation import run
result = run(DATA_NPZ, OUTPUT_DIR, MANIFEST_CSV, seed=20261011, epochs=35, batch_size=32)
result


In [ ]:
# Optional multi-seed comparison: E0 and E3 can be repeated only after the fixed protocol is accepted.
# for seed in (20261011, 20261012, 20261013):
#     run(DATA_NPZ, f'{OUTPUT_DIR}/seed_{seed}', MANIFEST_CSV, seed=seed, epochs=35, batch_size=32)
# Aggregate these separately; do not select only the best seed.


In [ ]:
# Inspect actual output and create a portable result bundle.
import json, shutil, pandas as pd
display(pd.read_csv(f'{OUTPUT_DIR}/ablation_results.csv'))
with open(f'{OUTPUT_DIR}/detailed_reports.json') as f: reports = json.load(f)
print(json.dumps({k: v['macro_f1'] for k, v in reports['reports'].items()}, indent=2))
shutil.make_archive('/content/airway_hybrid_ablation', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/airway_hybrid_ablation.zip')
